In [ ]:
from datetime import timedelta

import polars as pl
import polars.selectors as cs
import plotly.express as px

from aare_train.evaluation.evaluation import get_run_metrics, add_base_errors, join_start_end
from aare_train.params import read_params
from aare_train.paths import DATA_FOLDER, METRICS_FOLDER

# Accuracy difference between measurement eval and inference

Since we are using measurement data as training, validation and test data, the model not only takes the perfect accuracy of the air temperature for granted, but also our evaluation metrics are calculated in the most optimal situation.
In reality, the air temperature future covariate are weather forecasts from MeteoTest. They don't state how accurate their forecasts are, but since we historize everything, we can check ourselves. \
Additionally, we would like to correct our test set eval. The test set eval is done to get an estimate of how well we can expect the model to perform in the real world after deployment.
However, since we have a misalignment of test data and real data, we must assume that our calculated 'expected' accuracy is very optimistic and higher than it will actually perform.
How much worse it will actually perform depends on how good the forecasts of MeteoTest are (= how big the misalignment is).
To correct for this, we can take forecasts the model prototype has made so far and simulate a test set eval for the same model in same time period.
Then calculate how much worse forecasts are and try to extrapolate that for other models and into summer.
We can assume that the difference gets bigger as we transition from winter to summer and the further we forecast, since weather forecasts most likely also struggle more during those times than for example with short-term winter forecasts.
This means inaccuracies will compound and our forecasts 3-4 days into the future could be completely unusable (that also why the MVP only does 24h max).

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
params = read_params()
tz = params["general"]["timezone"]

In [ ]:
df_inf_meta = pl.read_csv(DATA_FOLDER / "backups/forecast_meta.csv").with_columns(
    pl.col("run_ts", "finished_at").str.to_datetime(time_zone=tz, time_unit="ns")
)
df_inf_meta

In [ ]:
# in our case here, there is only this single unique model, so no need to filter for the specific model.
# mlflow_run_id instead of model_version because this was back when the model was named "LR-dev".
df_inf_meta.select(pl.col("mlflow_run_id").unique())

In [ ]:
df_inf = pl.read_csv(DATA_FOLDER / "backups/forecast.csv")
df_inf = df_inf.with_columns(cs.string().str.to_datetime(time_zone=tz, time_unit="ns"))  # parquet reads ns
df_inf = df_inf.rename({"temp_bern": "pred"})
df_inf

In [ ]:
df_eval = pl.read_parquet(DATA_FOLDER / "metrics/raw/LR-dev-quasi-prod-test.parquet")
df_eval

In [ ]:
common_start = max(df_inf.select(pl.min("run_ts")).item(), df_eval.select(pl.min("run_ts")).item())
common_start

In [ ]:
common_end = min(df_inf.select(pl.max("run_ts")).item(), df_eval.select(pl.max("run_ts")).item())
common_end

In [ ]:
def assimilate(df: pl.DataFrame):
    df = df.lazy()
    run_ts = pl.col("run_ts")
    # filter away data that's definitely irrelevant
    df = df.filter(run_ts >= common_start, run_ts <= common_end)
    # only keep the latest run for each hour, since inference does 4 per hour. eval only has 1 per hour anyway.
    df = df.filter(run_ts.dt.minute() >= 40)  # inf does 00,15,30,45 (+ a small delta)
    # get the hour of the run_ts (should always be the first predicted timestamp of the run - 1h).
    # Note that I tried doing this with 'dt.replace', but that creates a new timestamp in the context of the timezone without
    # context of the original point in global time, so it thinks it's an ambiguous timestamp if we land on a
    # daylight savings hour. So either go UTC, replace and go back to CET, or (what I thought of earlier) subtract the hours, min and sec.
    # df = df.with_columns(run_hour=pl.col("run_ts").dt.replace(minute=0, second=0, microsecond=0))
    df = df.with_columns(
        run_hour=run_ts
        - pl.duration(minutes=run_ts.dt.minute(), seconds=run_ts.dt.second(), microseconds=run_ts.dt.microsecond())
    )

    df = df.collect()

    return df

In [ ]:
df_inf = assimilate(df_inf)
df_eval = assimilate(df_eval)

In [ ]:
df_inf

In [ ]:
df_eval

In [ ]:
df_inf = df_inf.join(df_eval.select("run_hour", "time", "actual"), on=["run_hour", "time"], validate="1:1")
df_inf

In [ ]:
# make sure that all times have the same ground truth, since that's not dependent on run_ts
df_inf.select(pl.col("actual").n_unique().eq(1).over("time").all())

In [ ]:
df_inf_pd = df_inf.to_pandas()
# use the same functions as in other eval scripts
add_base_errors(df_inf_pd)
metric_df_inf_pd = get_run_metrics(df_inf_pd)
metric_df_inf_pd = join_start_end(metric_df_inf_pd, df_inf_pd)
df_inf = pl.from_pandas(df_inf_pd)  # keep polars in sync
df_inf_pd

In [ ]:
df_eval_pd = df_eval.to_pandas()
metric_df_eval_pd = get_run_metrics(df_eval_pd)
metric_df_eval_pd = join_start_end(metric_df_eval_pd, df_eval_pd)

In [ ]:
metric_df_inf = pl.from_pandas(metric_df_inf_pd)
metric_df_eval = pl.from_pandas(metric_df_eval_pd)

In [ ]:
def get_summary(df: pl.DataFrame):
    return df.select(cs.float().median(), cs.float().std().name.suffix("_std"))

In [ ]:
get_summary(metric_df_eval)

In [ ]:
get_summary(metric_df_inf)

In [ ]:
# oh oh, up to 25% increased error in production vs test evaluation
get_summary(metric_df_inf) / get_summary(metric_df_eval)

In [ ]:
# to get a more accurate understanding of how bad it actually is, evaluate over season and lags

In [ ]:
# export both to visualize them in the existing marimo report

In [ ]:
df_inf.write_parquet(METRICS_FOLDER / "raw" / "LR-dev-live-proto.parquet")
df_eval.write_parquet(METRICS_FOLDER / "raw" / "LR-dev-live-proto-test.parquet")

In [ ]:
# from here on, don't use pandas
df_inf_pd = None
df_eval_pd = None
metric_df_inf_pd = None
metric_df_eval_pd = None

In [ ]:
def prep_raw(df: pl.DataFrame):
    return (
        df.lazy()
        .sort("run_ts", "time")
        # add lag column
        .with_columns(lag=((pl.col("time") - pl.col("run_ts")) / timedelta(hours=1) + 1).cast(int))
        # add absolute error columns
        .with_columns(ae=pl.col("err").abs(), adpd=pl.col("dpd").abs())
        # add unique integer id for each forecast, leave this for now because bad crossref potential
        # .with_columns(pl.col("run_ts").rle_id().alias("fc_i"))
        .collect()
    )

In [ ]:
df_eval = prep_raw(df_eval)
df_inf = prep_raw(df_inf)

In [ ]:
df_eval

In [ ]:
df_inf

In [ ]:
def get_run_metrics_filtered(df: pl.DataFrame):
    return (
        df.lazy()
        # what's actually displayed in aare.guru
        .filter(pl.col("lag") <= 14, pl.col("time").dt.hour() >= 7, pl.col("time").dt.hour() <= 21)
        .group_by("run_hour")
        .agg(
            pl.col("err").abs().mean().alias("mae"),
            pl.col("dpd").abs().mean().alias("madpd"),
        )
        .sort("run_hour")
        .collect()
    )

    return px.line(df, x="run_hour", y=["mae"])


# todo actually probably makes more sense to calculate diff earlier then aggregate by run and by lag after.
# todo also i'm an idiot, obviously doing this on the errors directly doesn't make sense. do it on both preds, then you
# can also get a useful percentage error and see if it rises over time.
# todo charts like in model evaluation, overlap colors
mae_run_diff = (
    get_run_metrics_filtered(df_eval)
    .select("run_hour", pl.col("mae").alias("mae_eval"))
    .join(get_run_metrics_filtered(df_inf).select("run_hour", pl.col("mae").alias("mae_inf")), on="run_hour")
    .with_columns((pl.col("mae_inf") - pl.col("mae_eval")).alias("diff"))
    .with_columns((pl.col("mae_inf") / pl.col("mae_eval")).alias("diff_p"))
)

px.line(
    mae_run_diff.rolling("run_hour", period="7d").agg(cs.float().mean()),
    x="run_hour",
    y=["mae_eval", "mae_inf", "diff", "diff_p"],
)

In [ ]:
mae_run_diff.median()